## Violin-plot visualisation — Segregation & Integration per Yeo-7 network

Each metric gets its own figure with **7 subplots** (one per Yeo network).  
The x-axis shows sessions (or Session_Run labels) and each violin shows the  
across-subject distribution at that time-point, with a horizontal line for the median.

**Functions provided:**
- `plot_violin_per_network_runs` — x-axis = every Session×Run label
- `plot_violin_per_network_sessions` — x-axis = session (runs averaged within each session)

Both accept the same `all_data` / `yeo_names` structure used in the line-plot notebook.

In [ ]:
import os
FIGURES_DIR = "../../../Results/Tessnim/figures/"
os.makedirs(FIGURES_DIR, exist_ok=True)
import sys, pathlib
project_root = pathlib.Path.cwd().parent
sys.path.append(str(project_root))

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

DATA_ROOT = "../../../Data/"
df_export = pd.read_csv("../../../Results/Tessnim/all_subjects_yeo7_metrics.csv")

YEO_NAMES = [
    "Visual",
    "Somatomotor",
    "Dorsal Attention",
    "Ventral Attention",
    "Limbic",
    "Frontoparietal",
    "Default",
]

all_data = {}

grouped = df_export.groupby(["subject", "session", "run", "network"])

tmp = {}

for (sid, sess_id, run_key, net_id), g in grouped:
    tmp.setdefault(sid, {}).setdefault(sess_id, {}).setdefault(run_key, {
        "condition": g["condition"].iloc[0],
        "segregation": np.full(7, np.nan),
        "integration": np.full(7, np.nan),
        "normalized_segregation": np.full(7, np.nan),
    })
    idx = int(net_id) - 1
    tmp[sid][sess_id][run_key]["segregation"][idx] = g["segregation"].mean()
    tmp[sid][sess_id][run_key]["integration"][idx] = g["integration"].mean()
    tmp[sid][sess_id][run_key]["normalized_segregation"][idx] = g["normalized_segregation"].mean()

for sid, sess_dict in tmp.items():
    subj = all_data.setdefault(sid, {"sessions": {}})
    for sess_id, run_dict in sess_dict.items():
        sess_runs = subj["sessions"].setdefault(sess_id, {})
        for run_key, rd in run_dict.items():
            sess_runs[run_key] = rd

print(f"Subjects loaded: {sorted(all_data.keys())}")

## Helper function — collect rows

In [ ]:
def collect_rows(all_data, condition_filter=None):
    """
    Build a list of row-dicts from all subjects.
    Each row: subject, session, run, condition,
              segregation (7,), integration (7,), normalized_segregation (7,).
    """
    all_rows = {}
    for sid, subj in all_data.items():
        rows = []
        for sess_id, sess_runs in subj["sessions"].items():
            for run_key in sorted(sess_runs.keys(), key=lambda x: int(x.replace("Run", ""))):
                rd = sess_runs[run_key]
                if condition_filter and rd["condition"] != condition_filter:
                    continue
                rows.append({
                    "subject":                sid,
                    "session":                sess_id,
                    "run":                    run_key,
                    "label":                  f"{sess_id}_{run_key}",
                    "condition":              rd["condition"],
                    "segregation":            rd["segregation"],
                    "integration":            rd["integration"],
                    "normalized_segregation": rd["normalized_segregation"],
                })
        all_rows[sid] = rows
    return all_rows

print("collect_rows() defined.")

## Violin-plot functions

In [ ]:
# ---------------------------------------------------------------------------
# Shared styling
# ---------------------------------------------------------------------------
VIOLIN_COLOR   = "#a8c4e0"   # steel-blue fill  (matches reference)
VIOLIN_EDGE    = "#1f6fad"   # darker edge
MEDIAN_COLOR   = "#1f6fad"   # horizontal median line
N_COLS         = 4           # subplots per row  (4 × 2 grid for 7 networks)


def _draw_violin_axes(
    ax,
    x_labels,
    data_matrix,       # shape (n_labels, n_subjects)  – may contain NaN
    network_name,
    ylabel,
    hline_zero=False,
):
    """
    Draw one violin-plot panel on `ax`.

    Parameters
    ----------
    ax           : matplotlib Axes
    x_labels     : list of str  – tick labels on the x-axis
    data_matrix  : 2-D array (n_labels × n_subjects); may contain NaN
    network_name : str – subplot title
    ylabel       : str – y-axis label
    hline_zero   : bool – draw a dashed line at y=0 (for normalised segregation)
    """
    n_labels = len(x_labels)
    positions = np.arange(n_labels)

    # Build a list of clean (non-NaN) arrays, one per x position
    datasets = []
    for j in range(n_labels):
        col = data_matrix[j, :]
        col = col[~np.isnan(col)]
        datasets.append(col)

    # Only draw a violin where we have ≥2 data points
    valid_pos  = [positions[j] for j, d in enumerate(datasets) if len(d) >= 2]
    valid_data = [d              for d in datasets              if len(d) >= 2]

    if valid_data:
        parts = ax.violinplot(
            valid_data,
            positions=valid_pos,
            widths=0.7,
            showmeans=False,
            showmedians=False,   # we draw our own styled median
            showextrema=False,
        )
        for body in parts["bodies"]:
            body.set_facecolor(VIOLIN_COLOR)
            body.set_edgecolor(VIOLIN_EDGE)
            body.set_linewidth(0.8)
            body.set_alpha(0.85)

    # Draw median lines
    for j, pos in enumerate(valid_pos):
        med = np.nanmedian(valid_data[j])
        ax.plot(
            [pos - 0.25, pos + 0.25],
            [med, med],
            color=MEDIAN_COLOR,
            linewidth=1.8,
            solid_capstyle="round",
        )

    # Scatter single-point positions (only 1 subject)
    single_pos  = [positions[j] for j, d in enumerate(datasets) if len(d) == 1]
    single_vals = [d[0]          for d in datasets               if len(d) == 1]
    if single_pos:
        ax.scatter(single_pos, single_vals, color=MEDIAN_COLOR, s=20, zorder=3)

    if hline_zero:
        ax.axhline(0, linestyle="--", linewidth=0.8, color="grey")

    ax.set_title(network_name, fontsize=9, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=7)
    ax.set_xticks(positions)
    ax.set_xticklabels(x_labels, rotation=90, fontsize=6)
    ax.tick_params(axis="y", labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)


# ---------------------------------------------------------------------------
# Public function 1: x-axis = every Session × Run
# ---------------------------------------------------------------------------
def plot_violin_per_network_runs(
    all_data,
    yeo_names,
    condition_filter=None,
    metrics=("segregation", "integration", "normalized_segregation"),
    figsize_per_panel=(2.8, 3.5),
):
    """
    One figure per metric, 7 subplots (one per Yeo network).
    X-axis = every Session_Run label; violins show across-subject distributions.

    Parameters
    ----------
    all_data         : dict  – same structure as in the line-plot notebook
    yeo_names        : list[str]  – 7 network names
    condition_filter : str or None
    metrics          : tuple of metric keys to plot
    figsize_per_panel: (width, height) per subplot in inches
    """
    subject_rows = collect_rows(all_data, condition_filter)
    cond_str = f" [{condition_filter}]" if condition_filter else ""

    # Union of all Session_Run labels, sorted
    all_labels_set = set()
    for rows in subject_rows.values():
        for r in rows:
            all_labels_set.add(r["label"])
    all_labels = sorted(all_labels_set)
    n_labels = len(all_labels)
    label_to_idx = {lbl: i for i, lbl in enumerate(all_labels)}

    n_subjects = len(subject_rows)

    # Accumulate arrays: metric -> network -> (n_labels, n_subjects)
    metric_keys = {
        "segregation":            "Segregation",
        "integration":            "Integration",
        "normalized_segregation": "Norm. Segregation",
    }

    # Build data cubes: metric → shape (n_labels, n_subjects, 7)
    cubes = {m: np.full((n_labels, n_subjects, 7), np.nan) for m in metric_keys}

    for s_idx, (sid, rows) in enumerate(subject_rows.items()):
        for r in rows:
            li = label_to_idx[r["label"]]
            cubes["segregation"][li, s_idx, :]            = r["segregation"]
            cubes["integration"][li, s_idx, :]            = r["integration"]
            cubes["normalized_segregation"][li, s_idx, :] = r["normalized_segregation"]

    n_nets = 7
    n_cols = min(N_COLS, n_nets)
    n_rows = int(np.ceil(n_nets / n_cols))

    fw = figsize_per_panel[0] * n_cols
    fh = figsize_per_panel[1] * n_rows

    for mkey in metrics:
        if mkey not in metric_keys:
            continue
        mlabel  = metric_keys[mkey]
        hline   = (mkey == "normalized_segregation")
        cube    = cubes[mkey]   # (n_labels, n_subjects, 7)

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(fw, fh),
            squeeze=False,
        )
        fig.suptitle(
            f"{mlabel} — per network, all runs{cond_str}",
            fontsize=11, fontweight="bold", y=1.01,
        )

        for net_i in range(n_nets):
            row_i = net_i // n_cols
            col_i = net_i  % n_cols
            ax = axes[row_i][col_i]

            # data_matrix shape: (n_labels, n_subjects)
            data_matrix = cube[:, :, net_i]

            _draw_violin_axes(
                ax,
                x_labels=all_labels,
                data_matrix=data_matrix,
                network_name=yeo_names[net_i],
                ylabel=mlabel,
                hline_zero=hline,
            )

        # Hide any unused subplot panels
        for net_i in range(n_nets, n_rows * n_cols):
            row_i = net_i // n_cols
            col_i = net_i  % n_cols
            axes[row_i][col_i].set_visible(False)

        plt.tight_layout()
        plt.show()


# ---------------------------------------------------------------------------
# Public function 2: x-axis = session (runs averaged within each session)
# ---------------------------------------------------------------------------
def plot_violin_per_network_sessions(
    all_data,
    yeo_names,
    condition_filter=None,
    metrics=("segregation", "integration", "normalized_segregation"),
    figsize_per_panel = (5.5, 3.5)   # best for PowerPoint,
):
    """
    Same as plot_violin_per_network_runs but the x-axis is sessions,
    and for each subject the runs are averaged within each session first.
    """
    subject_rows = collect_rows(all_data, condition_filter)
    cond_str = f" [{condition_filter}]" if condition_filter else ""

    # Union of sessions, sorted
    all_sessions_set = set()
    for rows in subject_rows.values():
        for r in rows:
            all_sessions_set.add(r["session"])
    all_sessions = sorted(all_sessions_set)
    n_sessions = len(all_sessions)
    sess_to_idx = {s: i for i, s in enumerate(all_sessions)}

    n_subjects = len(subject_rows)

    metric_keys = {
        "segregation":            "Segregation",
        "integration":            "Integration",
        "normalized_segregation": "Norm. Segregation",
    }

    # Build data cubes after session-averaging per subject
    # shape: (n_sessions, n_subjects, 7)
    cubes = {m: np.full((n_sessions, n_subjects, 7), np.nan) for m in metric_keys}

    for s_idx, (sid, rows) in enumerate(subject_rows.items()):
        # Group rows by session
        sess_groups = defaultdict(list)
        for r in rows:
            sess_groups[r["session"]].append(r)

        for sess, sess_rows in sess_groups.items():
            si = sess_to_idx[sess]
            cubes["segregation"][si, s_idx, :] = np.nanmean(
                np.vstack([r["segregation"] for r in sess_rows]), axis=0)
            cubes["integration"][si, s_idx, :] = np.nanmean(
                np.vstack([r["integration"] for r in sess_rows]), axis=0)
            cubes["normalized_segregation"][si, s_idx, :] = np.nanmean(
                np.vstack([r["normalized_segregation"] for r in sess_rows]), axis=0)

    n_nets = 7
    n_cols = min(N_COLS, n_nets)
    n_rows = int(np.ceil(n_nets / n_cols))

    fw = figsize_per_panel[0] * n_cols
    fh = figsize_per_panel[1] * n_rows

    for mkey in metrics:
        if mkey not in metric_keys:
            continue
        mlabel = metric_keys[mkey]
        hline  = (mkey == "normalized_segregation")
        cube   = cubes[mkey]

        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(fw, fh),
            squeeze=False,
        )
        fig.suptitle(
            f"{mlabel} — per network, session averages{cond_str}",
            fontsize=11, fontweight="bold", y=1.01,
        )

        for net_i in range(n_nets):
            row_i = net_i // n_cols
            col_i = net_i  % n_cols
            ax = axes[row_i][col_i]

            data_matrix = cube[:, :, net_i]  # (n_sessions, n_subjects)

            _draw_violin_axes(
                ax,
                x_labels=all_sessions,
                data_matrix=data_matrix,
                network_name=yeo_names[net_i],
                ylabel=mlabel,
                hline_zero=hline,
            )

        for net_i in range(n_nets, n_rows * n_cols):
            row_i = net_i // n_cols
            col_i = net_i  % n_cols
            axes[row_i][col_i].set_visible(False)

        plt.tight_layout()
        plt.show()


print("Violin-plot functions defined.")

## All runs — all conditions

In [ ]:
plot_violin_per_network_runs(all_data, YEO_NAMES)

## All runs — Feedback condition only

In [ ]:
plot_violin_per_network_runs(all_data, YEO_NAMES, condition_filter="Feedback")

## Session averages — all conditions

In [ ]:
plot_violin_per_network_sessions(all_data, YEO_NAMES)

## Session averages — Feedback condition only

In [ ]:
plot_violin_per_network_sessions(all_data, YEO_NAMES, condition_filter="Feedback")

---
### Tips for customisation

| What to change | Where |
|---|---|
| Violin fill / edge colour | `VIOLIN_COLOR` / `VIOLIN_EDGE` at the top of the functions cell |
| Median line colour | `MEDIAN_COLOR` |
| Number of columns in the grid | `N_COLS` |
| Panel size | `figsize_per_panel` kwarg |
| Which metrics to plot | `metrics` kwarg, e.g. `metrics=("segregation",)` |
| Condition filter | `condition_filter` kwarg: `"Feedback"`, `"Transfer"`, `"NoFeedback"`, or `None` for all |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ---------------------------------------------------------------------------
# Simple network colors
# ---------------------------------------------------------------------------
NETWORK_COLORS = [
    "#4C78A8",  # 1
    "#F58518",  # 2
    "#54A24B",  # 3
    "#E45756",  # 4
    "#72B7B2",  # 5
    "#B279A2",  # 6
    "#FF9DA6",  # 7
]


# ---------------------------------------------------------------------------
# Simple violin drawer
# ---------------------------------------------------------------------------
def _draw_violin_axes(ax, x_labels, data_matrix, title, ylabel, color, hline_zero=False):
    """
    data_matrix: shape (n_labels, n_subjects)
    """
    positions = np.arange(len(x_labels))
    datasets = [data_matrix[j, ~np.isnan(data_matrix[j])] for j in range(len(x_labels))]

    valid_pos = [i for i, d in enumerate(datasets) if len(d) >= 2]
    valid_data = [d for d in datasets if len(d) >= 2]

    if valid_data:
        vp = ax.violinplot(
            valid_data,
            positions=valid_pos,
            widths=0.8,
            showmeans=False,
            showmedians=False,
            showextrema=False,
        )
        for body in vp["bodies"]:
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_linewidth(0.5)
            body.set_alpha(0.8)

    # median lines
    for pos, d in zip(valid_pos, valid_data):
        med = np.median(d)
        ax.plot([pos - 0.2, pos + 0.2], [med, med], color="black", lw=1.4)

    # single-point cases
    single_pos = [i for i, d in enumerate(datasets) if len(d) == 1]
    single_val = [d[0] for d in datasets if len(d) == 1]
    if single_pos:
        ax.scatter(single_pos, single_val, color="black", s=12, zorder=3)

    if hline_zero:
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_xticks(positions)

    # cleaner x labels for slides
    ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=8)
    ax.tick_params(axis="y", labelsize=9)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


# ---------------------------------------------------------------------------
# Centered 7-panel layout:
# top row:    0 1 2 3
# bottom row:   4 5 6
# ---------------------------------------------------------------------------
def _make_centered_7_axes(figsize=(20, 8), dpi=300):
    fig = plt.figure(figsize=figsize, dpi=dpi)
    gs = fig.add_gridspec(2, 12, wspace=0.5, hspace=0.45)

    axes_list = [
        fig.add_subplot(gs[0, 0:3]),
        fig.add_subplot(gs[0, 3:6]),
        fig.add_subplot(gs[0, 6:9]),
        fig.add_subplot(gs[0, 9:12]),
        fig.add_subplot(gs[1, 1:4]),
        fig.add_subplot(gs[1, 4:7]),
        fig.add_subplot(gs[1, 7:10]),
    ]
    return fig, axes_list


# ---------------------------------------------------------------------------
# Session-averaged plot with centered bottom row
# ---------------------------------------------------------------------------
def plot_violin_per_network_sessions(
    all_data,
    yeo_names,
    condition_filter=None,
    metrics=("segregation", "integration", "normalized_segregation"),
    figsize=(20, 8),
    dpi=300,
):
    subject_rows = collect_rows(all_data, condition_filter)
    cond_str = f" [{condition_filter}]" if condition_filter else ""

    all_sessions = sorted({r["session"] for rows in subject_rows.values() for r in rows})
    sess_to_idx = {s: i for i, s in enumerate(all_sessions)}

    n_sessions = len(all_sessions)
    n_subjects = len(subject_rows)
    n_nets = len(yeo_names)

    metric_labels = {
        "segregation": "Segregation",
        "integration": "Integration",
        "normalized_segregation": "Norm. Segregation",
    }

    cubes = {m: np.full((n_sessions, n_subjects, n_nets), np.nan) for m in metric_labels}

    for s_idx, (_, rows) in enumerate(subject_rows.items()):
        sess_groups = defaultdict(list)
        for r in rows:
            sess_groups[r["session"]].append(r)

        for sess, sess_rows in sess_groups.items():
            si = sess_to_idx[sess]
            cubes["segregation"][si, s_idx, :] = np.nanmean(
                np.vstack([r["segregation"] for r in sess_rows]), axis=0
            )
            cubes["integration"][si, s_idx, :] = np.nanmean(
                np.vstack([r["integration"] for r in sess_rows]), axis=0
            )
            cubes["normalized_segregation"][si, s_idx, :] = np.nanmean(
                np.vstack([r["normalized_segregation"] for r in sess_rows]), axis=0
            )

    for mkey in metrics:
        if mkey not in cubes:
            continue

        fig, axes_list = _make_centered_7_axes(figsize=figsize, dpi=dpi)
        fig.suptitle(
            f"{metric_labels[mkey]} - per network, session averages{cond_str}",
            fontsize=14,
            fontweight="bold",
            y=0.98,
        )

        for net_i in range(n_nets):
            ax = axes_list[net_i]
            _draw_violin_axes(
                ax=ax,
                x_labels=all_sessions,
                data_matrix=cubes[mkey][:, :, net_i],
                title=yeo_names[net_i],
                ylabel=metric_labels[mkey],
                color=NETWORK_COLORS[net_i],
                hline_zero=(mkey == "normalized_segregation"),
            )


        # ---- SAVE HERE ----
        filename = f"{mkey}_violin_sessions"
        if condition_filter:
            filename += f"_{condition_filter}"
        filename += ".png"

        fig.savefig(os.path.join(FIGURES_DIR, filename), dpi=dpi, bbox_inches="tight")

        plt.show()

In [ ]:
plot_violin_per_network_sessions(all_data, YEO_NAMES)

In [ ]:
plot_violin_per_network_sessions(all_data, YEO_NAMES, condition_filter="Feedback")